# Day 2 - Part 1: DNN 구조 설계 실습 과제

이 과제에서는 오늘 배운 DNN 구조 설계 기법들을 직접 적용하여 자신만의 분류 모델을 만들어봅니다.
배운 내용을 바탕으로 자유롭게 모델 구조를 실험하고, 그 성능을 평가해보세요.

`데이터셋:` 위스콘신 유방암 데이터셋 (수업에서 사용한 것과 동일)
`목표:` 튜토리얼에서 만든 `AdvancedClassifier` 이상의 성능을 내는 모델을 구축하는 것을 목표로 도전해보세요!

### Step 1: 필요 라이브러리 및 데이터 준비

먼저, 실습에 필요한 라이브러리를 임포트하고 데이터를 불러와 전처리를 수행합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# 데이터 준비
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
input_features = X_train.shape[1]
output_classes = 2

# Dataset 및 DataLoader 생성
class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = BreastCancerDataset(X_train, y_train)
test_dataset = BreastCancerDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 과제 1: 나만의 DNN 모델 설계하기

`MyClassifier` 클래스의 `__init__` 부분을 채워 자신만의 모델을 설계하세요.

`요구사항:`
1. `nn.Sequential`을 사용하여 네트워크를 구성하세요.
2. `3개 이상의 은닉층`을 포함해야 합니다.
3. 모든 은닉층에 `배치 정규화(BatchNorm)` 를 적용하세요.
4. 모든 은닉층에 `드롭아웃(Dropout)` 을 적용하세요. (드롭아웃 확률은 자유롭게 조절)
5. 모든 은닉층의 활성화 함수는 `ReLU`를 사용하세요.

In [ ]:
class MyClassifier(nn.Module):
    def __init__(self, num_features, num_classes):
        super(MyClassifier, self).__init__()
        
        # === YOUR CODE HERE === #
        # 예시: self.net = nn.Sequential(...)
        # 층의 깊이, 뉴런 수, 드롭아웃 확률 등을 자유롭게 변경해보세요.
        self.net = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )
        # ====================== #

    def forward(self, x):
        return self.net(x)

### 과제 2: 가중치 초기화 적용하기

모델 인스턴스를 생성하고, 모델의 모든 선형 계층(Linear Layer)에 `He 초기화`를 적용하는 코드를 작성하세요.
`model.apply()` 함수를 사용하는 것을 추천합니다.

In [ ]:
my_model = MyClassifier(input_features, output_classes)

def init_weights(module):
    # === YOUR CODE HERE === #
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
        if module.bias is not None:
            nn.init.constant_(module.bias, 0)
    # ====================== #

# 모델에 초기화 함수 적용
my_model.apply(init_weights)

print("모델 생성 및 가중치 초기화 완료!")
print(my_model)

### 과제 3: 모델 훈련 및 평가 코드 완성하기

아래 훈련 루프의 빈칸을 채워 모델을 훈련시키고, 테스트 데이터셋에 대한 최종 정확도를 계산하여 출력하세요.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(my_model.parameters(), lr=0.001)
num_epochs = 50

for epoch in range(num_epochs):
    my_model.train() # 훈련 모드 설정
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        
        # === YOUR CODE HERE (Training loop) === #
        # 1. 경사도 초기화
        optimizer.zero_grad()
        # 2. 순전파 (Forward pass)
        outputs = my_model(features)
        # 3. 손실 계산
        loss = criterion(outputs, labels)
        # 4. 역전파 (Backward pass)
        loss.backward()
        # 5. 파라미터 업데이트
        optimizer.step()
        # =================================== #

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("\nFinished Training!")

# --- 최종 평가 --- #
my_model.eval() # 평가 모드 설정
correct = 0
total = 0
with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)
        # === YOUR CODE HERE (Evaluation) === #
        # 1. 모델의 예측값 계산
        outputs = my_model(features)
        # 2. 가장 높은 확률을 가진 클래스를 예측 결과로 선택
        _, predicted = torch.max(outputs.data, 1)
        # ================================= #
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'\nTest Accuracy of the model: {accuracy:.2f} %')

### 과제 4 (심화): 하이퍼파라미터 실험

자신이 설계한 모델의 성능을 더 높이기 위해 다양한 실험을 진행해보세요.

- `구조 변경`: 층의 수나 뉴런의 수를 바꿔보세요. (예: 버섯 모양 구조, 다이아몬드 구조 등)
- `드롭아웃 확률 변경`: 드롭아웃 확률(p)을 0.1 ~ 0.7 사이에서 다양하게 조절해보세요.
- `옵티마이저 변경`: `optim.Adam` 대신 `optim.SGD`나 `optim.RMSprop`을 사용해보세요.
- `학습률(Learning Rate) 변경`: 옵티마이저의 `lr` 값을 조절해보세요.

어떤 조합이 가장 높은 테스트 정확도를 보였나요? 결과를 아래 마크다운 셀에 자유롭게 기록해보세요.

#### 나의 실험 결과

*(여기에 자신의 실험 내용과 결과를 자유롭게 작성하세요)*

`최고 성능 조합:`
- 구조: ...
- 드롭아웃 확률: ...
- 옵티마이저: ...
- 학습률: ...
`최고 정확도:` ... %